## Импорты библиотек

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

import optuna
import optuna.logging

optuna.logging.set_verbosity(optuna.logging.CRITICAL)

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
    mean_absolute_percentage_error,
)

from utils.utils import (
    connection,
    data_from_ticker,
    data_from_tpulse,
    data_from_macrofactors,
)
import utils.config_ml as config_ml

## Импорты глобальных переменных

In [27]:
companies = pd.read_sql("SELECT * FROM companies", connection())
tickers = companies['ticker']
left_date = '2026-01-12'
right_date = '2026-03-12'
# left_date = config_ml.LEFT_DATE
# right_date = config_ml.RIGHT_DATE
train_period = config_ml.TRAIN_PERIOD
val_period = config_ml.VAL_PERIOD
test_period = config_ml.TEST_PERIOD
step = config_ml.STEP
models = config_ml.MODELS
n_trials = config_ml.N_TRIALS
metric_optuna = config_ml.METRIC_OPTUNA
top_n_features = config_ml.TOP_N_FEATURES

## Сбор фичей по тикеру по дням

In [29]:
def pack_all_data_for_ml_models(ticker: str, left_date: str, right_date: str, conn):
    tpulse_data = data_from_tpulse(ticker, left_date, right_date, conn)
    ticker_data = data_from_ticker(ticker, left_date, right_date, conn)
    macrofactor_data = data_from_macrofactors(ticker, left_date, right_date, conn)

    data = tpulse_data.merge(macrofactor_data, how='left', on=['dt', 'ticker']).merge(ticker_data, how='left', on=['dt', 'ticker'])
    data = data[~data['target'].isnull()]

    return data

In [30]:
# pack_all_data_for_ml_models('SBER', '2025-07-01', '2025-09-01', connection())

In [31]:
# pack_all_data_for_ml_models('SBER', '2026-01-12', '2026-03-12', connection())

## Цикл ML

In [32]:
# Кросс-валидация скользящим окном
def sliding_windows_cross_validatin(df, train_days, val_days, test_days, step):
    windows = []
    n = len(df)
    current_test_end = n
    while True:
        test_start = current_test_end - test_days
        if test_start < 0:
            break
        val_start = test_start - val_days
        if val_start < 0:
            break
        train_start = val_start - train_days
        if train_start < 0:
            break
        windows.append({'train': df.iloc[train_start:val_start],
                        'val': df.iloc[val_start:test_start],
                        'test': df.iloc[test_start:current_test_end],
                        'id': current_test_end,
                        'dates': {
                                    'train': (df['dt'].iloc[train_start], df['dt'].iloc[val_start-1]),
                                    'val': (df['dt'].iloc[val_start], df['dt'].iloc[test_start-1]),
                                    'test': (df['dt'].iloc[test_start], df['dt'].iloc[current_test_end-1])
                                }
                        })
        current_test_end -= step
        if current_test_end < train_days + val_days + test_days:
            break
    return windows[::-1]

# Отбор топ фичей
def select_top_features(model, feature_names, top_n):
    if hasattr(model, 'coef_') and len(model.coef_) > 0:
        coefs = np.abs(model.coef_)
        top_indices = np.argsort(coefs)[-top_n:]
    elif hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        top_indices = np.argsort(importances)[-top_n:]
    else:
        return feature_names[:top_n]
    return [feature_names[i] for i in top_indices]

# Подбор гиперпараметров
def make_objective(model_name, X_train, y_train, X_val, y_val):
    def objective(trial):
        model_cfg = models[model_name]
        params = {}
        if 'optuna_objective' in model_cfg and callable(model_cfg['optuna_objective']):
            params = model_cfg['optuna_objective'](trial)
        if model_name == 'CatBoost':
            model = CatBoostRegressor(**params, verbose=False)
        else:
            model = model_cfg['model']
            if params:
                model.set_params(**params)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)

        if metric_optuna == 'MAPE':
            return mean_absolute_percentage_error(y_val, y_pred)
        elif metric_optuna == 'MAE':
            return mean_absolute_error(y_val, y_pred)
    return objective


In [33]:
# Цикл ML
def train_val_test_ml_models(df, windows, models, ticker):
    print(f'Запущен цикл разработки ML моделей для прогнозирования стоимости акций "{ticker}"\n')
    df_ml_db = pd.DataFrame(columns=['dt', 'ticker', 'left_date', 'right_date', 'model_name', 'model_value', 'train_period', 'val_period', 'test_period', 'step', 'mape_mean', 'mae_mean', 'best_features'])
    df_ml = pd.DataFrame(columns=['test_period', 'model_name', 'mape', 'mae'])
    for model in models:
        ml_info = [] # model, features, mape,
        for idx, window in enumerate(windows):
            train, val, test = window['train'], window['val'], window['test']
            feature_cols = [col for col in train.columns if col not in ['dt', 'ticker', 'target']]

            print(f'Окно {idx+1} / модель {model}')
            print(f"Обучение {len(window['train'])} дн. {window['dates']['train'][0]} - {window['dates']['train'][1]} / Валидация {len(window['val'])} дн. {window['dates']['val'][0]} - {window['dates']['val'][1]} / Тестирование {len(window['test'])} дн. {window['dates']['test'][0]} - {window['dates']['test'][1]}")

            if model in ['LinearRegression', 'DecisionTree', 'GradientBoosting']:
                train = train.fillna(0)
                val = val.fillna(0)
                test = test.fillna(0)

            # Деление на train/test/val
            X_train, y_train = train[feature_cols], train['target']
            X_val, y_val = val[feature_cols], val['target']
            X_test, y_test = test[feature_cols], test['target']

            # Для линейных моделей важно стандартизировать данные
            if model == 'LinearRegression':
                best_params = {}
                scaler = StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train)
                X_test_scaled = scaler.transform(X_test)
            else:
                # Для нелинейных моделей поиск ГП с помощью Optuna
                objective = make_objective(model, X_train, y_train, X_val, y_val)
                study = optuna.create_study(direction='minimize')
                study.optimize(objective, n_trials=4)
                best_params = study.best_params
                print(f"Лучшие ГП модели: {best_params} c {metric_optuna}={study.best_value:.3f}")

            if model == 'CatBoost':
                best_optuna_model = CatBoostRegressor(**best_params, verbose=False)
            else:
                best_optuna_model = models[model]['model']
                best_optuna_model.set_params(**best_params)
            if model == 'LinearRegression':
                best_optuna_model.fit(X_train_scaled, y_train)
            else:
                best_optuna_model.fit(X_train, y_train)

            # Отбор топ фичей
            selected_features = select_top_features(best_optuna_model, feature_cols, top_n_features)
            print(f'Отобрано топ-{top_n_features} фичей, 5 лучших из них: {selected_features[:5]}')
            selected_idx = [feature_cols.index(feature) for feature in selected_features]

            # Тестирование модели с лучшими ГП только на топ фичах
            if model == 'LinearRegression':
                best_optuna_model.fit(X_train_scaled[:, selected_idx], y_train)
                y_test_pred = best_optuna_model.predict(X_test_scaled[:, selected_idx])
            else:
                best_optuna_model.fit(X_train[selected_features], y_train)
                y_test_pred = best_optuna_model.predict(X_test[selected_features])
            test_mape = mean_absolute_percentage_error(y_test, y_test_pred)
            test_mae = mean_absolute_error(y_test, y_test_pred)
            print(f'Метрики лучшей модели в данном окне: MAPE={test_mape:.3f}, MAE={test_mae:.2f}\n')
            ml_info.append([best_optuna_model, selected_features, test_mape])

        # Определение лучшей модели - ГП из модели из окна с минимальным mape, лучшие фичи - те, которые есть во всех окнах
        best_model = sorted(ml_info, key=lambda x: x[-1])[0][0]
        best_features = set(ml_info[0][1])
        for feature_list in ml_info[1:]:
            best_features.intersection_update(feature_list[1])
        best_features = list(best_features)
        print('Формируется лучшая модель из построенных ранее...')
        print(f'Лучшая {model} модель - {best_model} с фичами в количестве {len(best_features)} шт.')
        mape_mean, mae_mean = [], []

        # Расчет статистик
        for idx, window in enumerate(windows):
            train, val, test = window['train'], window['val'], window['test']
            X_train, y_train = train[best_features], train['target']
            X_test, y_test = test[best_features], test['target']
            best_model.fit(X_train, y_train)
            best_model_pred = best_model.predict(X_test)
            test_mape = mean_absolute_percentage_error(y_test, best_model_pred)
            test_mae = mean_absolute_error(y_test, best_model_pred)
            mape_mean.append(test_mape)
            mae_mean.append(test_mae)
            new_row = pd.Series([f"{window['dates']['test'][0]} - {window['dates']['test'][1]}", model, round(test_mape,3), round(test_mae,3)], index=df_ml.columns)
            df_ml = pd.concat([df_ml, new_row.to_frame().T], ignore_index=True)
            print(f'Окно {idx+1}: лучшая {model} модель имеет MAPE={test_mape:.3f} ({round(100*(ml_info[idx][-1] - test_mape)/ml_info[idx][-1])}% к модели на окне {idx+1} ранее), MAE={test_mae:.2f}')

        new_row_db = pd.Series([datetime.now().strftime('%Y-%m-%d %H:%M:%S'), ticker, left_date, right_date, model, str(best_model), train_period, val_period, test_period, step, float(round(np.mean(mape_mean),3)), float(round(np.mean(mae_mean),3)), json.dumps(best_features)], index=df_ml_db.columns)
        df_ml_db = pd.concat([df_ml_db, new_row_db.to_frame().T], ignore_index=True)
        print('------------------------------------------------------------------------------------------------------------')

    # Отправить в бд созданные лучшие модели
    df_ml_db.to_sql('ml_models_data', con=connection(), if_exists='append', index=False)

    return df_ml.sort_values(by=['test_period', 'mae']), df_ml_db

In [34]:
for ticker in tickers:
    try:
        data = pack_all_data_for_ml_models(ticker, left_date, right_date, connection())

        df = data.copy()
        df.to_csv(fr'{os.getcwd()}/all_features_{ticker}_2026.csv', index=False)

        df = pd.read_csv(fr'{os.getcwd()}/all_features_{ticker}_2026.csv')
        df = df[~df['target'].isnull()]

        windows = sliding_windows_cross_validatin(df, train_period, val_period, test_period, step)
        df_ml, df_ml_db = train_val_test_ml_models(df, windows, models, ticker)
        
        print(df_ml)
        print(df_ml_db)
    except Exception as e:
        print(e)
        print(f"При работе функции произошла с ошибка со следующим тикером {ticker}")
        continue

Запущен цикл разработки ML моделей для прогнозирования стоимости акций "SBER"

Окно 1 / модель DecisionTree
Обучение 21 дн. 2026-01-16 - 2026-02-05 / Валидация 14 дн. 2026-02-06 - 2026-02-23 / Тестирование 7 дн. 2026-02-24 - 2026-03-02
Лучшие ГП модели: {'max_depth': 3, 'min_samples_split': 9, 'min_samples_leaf': 4, 'max_features': 'sqrt'} c MAPE=0.017
Отобрано топ-500 фичей, 5 лучших из них: ['RTSI_open_lag_9', 'RTSI_open_rm_8', 'RTSI_open_lag_8', 'RTSI_open_rm_7', 'RTSI_open_lag_7']
Метрики лучшей модели в данном окне: MAPE=0.036, MAE=11.39

Окно 2 / модель DecisionTree
Обучение 21 дн. 2026-01-23 - 2026-02-12 / Валидация 14 дн. 2026-02-17 - 2026-03-02 / Тестирование 7 дн. 2026-03-03 - 2026-03-11
Лучшие ГП модели: {'max_depth': 13, 'min_samples_split': 20, 'min_samples_leaf': 5, 'max_features': None} c MAPE=0.028
Отобрано топ-500 фичей, 5 лучших из них: ['RTSI_open_rm_9', 'RTSI_open_lag_9', 'RTSI_open_rm_8', 'RTSI_open_lag_8', 'RTSI_open_rm_7']
Метрики лучшей модели в данном окне: MAP